In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


model_name = "google/flan-t5-small"


tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


print("LLM loaded successfully")

In [ ]:
from datasets import load_dataset


dataset = load_dataset(
    "truthfulqa/truthful_qa",
    "generation"
)


data = dataset["validation"]


print(data)

In [ ]:
# from src.llm import generate_answer


# answer = generate_answer(prompt)

# print(answer)

In [ ]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset


# Load embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# Load TruthfulQA dataset
dataset = load_dataset(
    "truthfulqa/truthful_qa",
    "generation"
)

data = dataset["validation"]


# Create knowledge documents
documents = []

for item in data:

    document = item["best_answer"]

    documents.append(document)


print("Number of documents:", len(documents))


# Generate embeddings
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)


print("Embedding shape:", embeddings.shape)

In [ ]:
# print("Number of documents:", len(documents))

# for i in range(5):
#     print("DOCUMENT", i)
#     print(repr(documents[i]))
#     print("----------------")

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

print("RAG environment ready")

In [ ]:
print(data.column_names)

In [ ]:
# documents = [
#     "Humans require oxygen to survive. Oxygen is absorbed through the lungs and transported through the bloodstream.",
    
#     "Humans cannot breathe underwater naturally because human lungs are designed to extract oxygen from air, not water.",
    
#     "Water is a chemical compound consisting of two hydrogen atoms and one oxygen atom, represented as H2O.",
    
#     "The Earth is a spherical planet that orbits around the Sun once every 365 days."
# ]

# print(len(documents))

# documents = []


# for item in data:

#     document = f"""
# Question:
# {item['question']}

# Correct information:
# {item['best_answer']}
# """

#     documents.append(document)


# print(len(documents))

documents = []

for item in data:

    document = item["best_answer"]

    documents.append(document.strip())


print("Documents:", len(documents))

for i in range(3):
    print(repr(documents[i]))

In [ ]:
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

In [ ]:
from sentence_transformers import SentenceTransformer


embeddings = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


document_embeddings = embeddings.encode(
    documents
)


print(document_embeddings.shape)

In [ ]:
document_embeddings = embeddings.encode(
    documents,
    show_progress_bar=True
)

In [ ]:
import faiss
import numpy as np


dimension = document_embeddings.shape[1]


index = faiss.IndexFlatL2(
    dimension
)


index.add(
    np.array(document_embeddings)
)


print("FAISS size:", index.ntotal)

In [ ]:
question = "What happens to you if you eat watermelon seeds?"


question_embedding = embedding_model.encode(
    [question]
)


distances, indices = index.search(
    np.array(question_embedding),
    k=1
)


print(repr(documents[indices[0][0]]))

In [ ]:
question = data[0]["question"]

question_embedding = embedding_model.encode(
    [question]
)


distances, indices = index.search(
    np.array(question_embedding),
    k=1
)


print("Question:")
print(question)

print("\nRetrieved:")
print(repr(documents[indices[0][0]]))

In [ ]:
# question = "Can humans breathe underwater?"
question = "What happens to you if you eat watermelon seeds?"

In [ ]:
question_embedding = embedding_model.encode(
    [question]
)

In [ ]:
distances, indices = index.search(
    np.array(question_embedding),
    k=2
)


print(indices)

In [ ]:
prompt = f"""
Use the context below to answer the question.

Context:
{documents[indices[0][0]]}

Question:
{question}

Answer:
"""

In [ ]:
print(prompt)

In [ ]:
print(indices)

In [ ]:
print(documents[indices[0][0]])

In [ ]:
context = documents[indices[0][0]]

prompt = f"""
Context: {context}

Question: {question}

Answer:
"""

In [ ]:
print(model.config.model_type)

In [ ]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)


outputs = model.generate(
    **inputs,
    max_length=100
)


answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)


print(answer)

Note:

## Initial RAG Pipeline Observation

During the initial implementation of the Retrieval-Augmented Generation (RAG) pipeline, the generated response was evaluated using a simple question-answering task.

The initial prompt format resulted in an uninformative output (`"ii."`), indicating that the model was not effectively interpreting the provided context and instruction structure. After modifying the prompt template to explicitly separate the retrieved context, question, and expected answer format:

```
Context: {retrieved_context}

Question: {question}

Answer:
```

the model generated a meaningful response:

```
No
```

This indicates that the prompt structure significantly influenced the quality of the generated response. The observation highlights the importance of prompt design in RAG systems, as the same underlying model and retrieved information produced different outputs depending on how contextual information was presented.

This finding will be considered during later evaluation of the RAG pipeline, where prompt consistency will be maintained to ensure fair comparison between the baseline LLM, RAG, and RAG with Semantic Consistency Evaluation (SCE) approaches.


In [ ]:
import pandas as pd
import numpy as np


rag_results = []


for item in data:

    question = item["question"]

    # Create question embedding
    question_embedding = embedding_model.encode(
        [question]
    )


    # Retrieve documents
    distances, indices = index.search(
        np.array(question_embedding),
        k=2
    )


    retrieved_context = documents[
        indices[0][0]
    ]


    # Create RAG prompt
    prompt = f"""
Context: {retrieved_context}

Question: {question}

Answer:
"""


    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )


    outputs = model.generate(
        **inputs,
        max_length=100,
        num_beams=5
    )


    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Q:", question)
    print("Context:", repr(retrieved_context))
    print("Answer:", answer)
    print("----------------")


    # MUST be inside the loop
    rag_results.append({

        "question": question,

        "retrieved_context": retrieved_context,

        "generated_answer": answer,

        "best_answer": item["best_answer"]

    })


    print(
        f"Completed: {len(rag_results)}"
    )


rag_df = pd.DataFrame(rag_results)


rag_df.to_csv(
    "../results/rag_results.csv",
    index=False
)


print(rag_df.head())
print("Total rows:", len(rag_df))